In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv("cleaned_customer_churn.csv")

df.drop(['RowNumber', 'Surname'], axis=1, inplace=True)

X = df.drop(['Exited', 'CustomerId', 'Gender'], axis=1)
y = df['Exited']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = X.select_dtypes(include='object').columns
num_features = X.select_dtypes(exclude='object').columns

oh_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer([
    ("OneHotEncoder", oh_transformer, cat_features),
    ("StandardScaler", numeric_transformer, num_features)
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [6]:
oversample = SMOTE(random_state=42)
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [7]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [8]:
model_scores = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    model_train_accuracy = accuracy_score(y_train, y_train_pred)
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    model_train_precision = precision_score(y_train, y_train_pred, zero_division=0)
    model_train_recall = recall_score(y_train, y_train_pred, zero_division=0)
    model_train_roc_auc = roc_auc_score(y_train, y_train_prob)

    model_test_accuracy = accuracy_score(y_test, y_test_pred)
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    model_test_precision = precision_score(y_test, y_test_pred, zero_division=0)
    model_test_recall = recall_score(y_test, y_test_pred, zero_division=0)
    model_test_roc_auc = roc_auc_score(y_test, y_test_prob)

    model_scores.append({
        "Model": list(models.keys())[i],
        "Train Accuracy": model_train_accuracy,
        "Test Accuracy": model_test_accuracy,
        "Test F1": model_test_f1,
        "Test Precision": model_test_precision,
        "Test Recall": model_test_recall,
        "Test ROC-AUC": model_test_roc_auc
    })

    print(list(models.keys())[i], ":-")
    print("-"*35)
    print("Model Performance for Training Set")
    print(f"Accuracy Score is: {model_train_accuracy}")
    print(f"F1 Score is: {model_train_f1}")
    print(f"Precision is: {model_train_precision}")
    print(f"Recall is: {model_train_recall}")
    print(f"ROC AUC score is: {model_train_roc_auc}")
    print("-"*35)
    print("Model Performance for Testing Set")
    print(f"Accuracy Score is: {model_test_accuracy}")
    print(f"F1 Score is: {model_test_f1}")
    print(f"Precision is: {model_test_precision}")
    print(f"Recall is: {model_test_recall}")
    print(f"ROC AUC score is: {model_test_roc_auc}")
    print("="*35)
    print("\n")

Random Forest :-
-----------------------------------
Model Performance for Training Set
Accuracy Score is: 1.0
F1 Score is: 1.0
Precision is: 1.0
Recall is: 1.0
ROC AUC score is: 1.0
-----------------------------------
Model Performance for Testing Set
Accuracy Score is: 0.8445
F1 Score is: 0.8415276519129783
Precision is: 0.6304347826086957
Recall is: 0.5700245700245701
ROC AUC score is: 0.8374144560585238


Decision Tree :-
-----------------------------------
Model Performance for Training Set
Accuracy Score is: 1.0
F1 Score is: 1.0
Precision is: 1.0
Recall is: 1.0
ROC AUC score is: 1.0
-----------------------------------
Model Performance for Testing Set
Accuracy Score is: 0.7725
F1 Score is: 0.7795130227257384
Precision is: 0.45041322314049587
Recall is: 0.5356265356265356
ROC AUC score is: 0.68432299788232


Logistic Regression :-
-----------------------------------
Model Performance for Training Set
Accuracy Score is: 0.7081632653061225
F1 Score is: 0.7081036946315575
Precision i

In [9]:
model_comparison = pd.DataFrame(model_scores)
model_comparison.sort_values(by='Test F1', ascending=False)

,Model,Train Accuracy,Test Accuracy,Test F1,Test Precision,Test Recall,Test ROC-AUC
3,Gradient Boosting,0.875039,0.8410,0.843458,0.600451,0.653563,0.860311
0,Random Forest,1.000000,0.8445,0.841528,0.630435,0.570025,0.837414
1,Decision Tree,1.000000,0.7725,0.779513,0.450413,0.535627,0.684323
2,Logistic Regression,0.708163,0.7160,0.740075,0.388658,0.690418,0.774275


In [10]:
best_model_name = model_comparison.sort_values(
    by='Test F1', ascending=False
).iloc[0]['Model']

final_model = models[best_model_name]

print("Selected Model:", best_model_name)

Selected Model: Gradient Boosting


In [11]:
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      1593
           1       0.60      0.65      0.63       407

    accuracy                           0.84      2000
   macro avg       0.75      0.77      0.76      2000
weighted avg       0.85      0.84      0.84      2000



In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

Accuracy: 0.841
Precision: 0.600451467268623
Recall: 0.6535626535626535
F1 Score: 0.8434584873949581
ROC-AUC: 0.8603110043788009


In [14]:
import joblib
import os

In [15]:
joblib.dump(final_model, "churn_model.pkl")

joblib.dump(preprocessor, "preprocessor.pkl")

joblib.dump(X.columns.tolist(), "feature_columns.pkl")

print("Model and preprocessing objects saved successfully!")

Model and preprocessing objects saved successfully!
